# Урок 34 — Asyncio: Кооперативна Багатозадачність та Архітектура Event Loop

## yield · Coroutines · Tasks · await · gather · Backend Concurrency

**Модуль 4 · Network & Concurrent Systems · Автор: Viktor Nikoriak**

---

# Про що цей урок?

У попередньому уроці ми вивчили `multiprocessing` — справжній паралелізм через окремі процеси ОС.

Тепер питання: *як Python обробляє 10 000 мережевих з'єднань на одному потоці?*

Відповідь: **`asyncio`** — однопотокова кооперативна багатозадачність.

Але перш ніж вивчити `asyncio`, нам потрібно зрозуміти **його фундамент** — механізм,
на якому побудований весь async Python.

---

## Структура уроку

| Частина | Тема | Ключові концепції |
|---------|------|-------------------|
| **I** | `yield` та Класичні Корутини | `yield`, генератори, `.send()`, `yield from` |
| **II** | Сучасний Asyncio | Event Loop, `async/await`, `Task`, `gather` |

---

## Але ось що дивує більшість розробників:

```python
async def fetch_data():
    print("Starting...")
    await asyncio.sleep(1)
    print("Finished")

fetch_data()  # Що виведе?
```

Якщо ти відповів "Starting..." — цей урок саме для тебе.

---

**Цей урок руйнує 5 хибних уявлень:**

| # | Хибне уявлення | Реальність |
|---|----------------|------------|
| 1 | `async def` запускає функцію в фоні | Виклик повертає coroutine object, нічого не виконує |
| 2 | `asyncio.gather()` робить код паралельним | Concurrency, а не parallelism — один потік |
| 3 | `async def` з `time.sleep()` — асинхронний | `time.sleep()` блокує весь Event Loop |
| 4 | `requests.get()` в `async def` — OK | Синхронний виклик заморожує всі з'єднання |
| 5 | CPU-важкі задачі можна `await`-ати | Без `await` Event Loop не отримує контроль |

# Налаштування середовища

In [ ]:
# Встановлення залежностей для Jupyter
%pip install nest_asyncio --quiet

import nest_asyncio
nest_asyncio.apply()  # Дозволяє asyncio.run() всередині Jupyter

import sys
sys.path.insert(0, ".")

# Імпорт допоміжних корутин
from _coroutines_34 import (
    run, lifecycle_demo, blocking_demo, nonblocking_demo,
    gather_demo, safe_gather_demo, connection_pool_demo,
    cpu_broken_demo, cpu_fixed_demo,
)

print("✅ nest_asyncio активовано — asyncio.run() працює в Jupyter")
print("✅ Корутини імпортовано з _coroutines_34.py")


---

# Частина I — Фундамент: `yield` та Класичні Корутини

> Щоб дійсно зрозуміти `asyncio`, `await` і Event Loop — потрібно спочатку
> зрозуміти `yield`. Це не просто синтаксис: це механізм призупинення
> виконання функції із збереженням стану, на якому побудовано весь async Python.

---

## Ключова ідея: return vs yield

```
def звичайна():            def з_yield():
    x = 10                     x = 10
    return x   ← знищує       yield x    ← тимчасово зупиняє
               локальний стан             зберігає стан
```

| `return` | `yield` |
|----------|---------|
| Передає значення і завершує функцію | Передає значення і **призупиняє** функцію |
| Локальні змінні знищуються | Локальні змінні **збережені** у пам'яті |
| Повторний виклик — нова функція | Повторне продовження — з тієї ж точки |
| Стан: CLOSED | Стан: SUSPENDED → RUNNING → SUSPENDED |

## Стани об'єкта генератора / корутини

```
gen = my_generator()          ← CREATED  (код не виконується!)
next(gen)                     ← RUNNING  (до першого yield)
# yield X                     ← SUSPENDED (стан збережено)
next(gen)                     ← RUNNING  (відновлення)
# StopIteration               ← CLOSED   (функція завершилась)
```

## Що зберігається у SUSPENDED стані?

```python
def counter(start):
    x = start          # ← локальна змінна збережена
    while True:
        yield x        # ← точка збереження: знаємо 'x' і 'start'
        x += 1         # ← продовжиться звідси при next()
#                                       ↑
#                          frame object у heap пам'яті (~1 KB)
#                          містить: locals(), f_lasti (pointer)
```

In [ ]:
import inspect

def counter(start=0):
    """Генератор: yield ВИРОБЛЯЄ значення."""
    x = start
    while True:
        yield x
        x += 1

gen = counter(10)

# Стан одразу після створення
print(f"Після створення:   {inspect.getgeneratorstate(gen)}")

# Перший виклик: код виконується до yield
val = next(gen)
print(f"Після next():      {inspect.getgeneratorstate(gen)}")
print(f"Значення:          {val}")

# Другий виклик: відновлення ПІСЛЯ yield, потім x+=1, потім знову yield
val2 = next(gen)
print(f"Після next() x2:   {val2}  (x збережено між викликами!)")

val3 = next(gen)
print(f"Після next() x3:   {val3}")

print()
print("Ключове: x зберігається між next() — це і є збереження стану")

## Що відбувається під капотом: frame object

```
Пам'ять Python (heap):
┌─────────────────────────────────────────────┐
│  generator object  (counter instance)       │
│                                             │
│  gi_frame (frame object):                  │
│    ├── f_locals: {"start": 10, "x": 12}    │ ← змінні збережені!
│    ├── f_lasti:  байткод pointer → yield   │ ← де зупинились
│    └── f_code:   bytecode of counter()     │
│                                             │
│  gi_running: False  (зараз SUSPENDED)       │
│  gi_yieldfrom: None                         │
└─────────────────────────────────────────────┘
```

Кожен раз коли ти пишеш `next(gen)` → Python відновлює цей frame,
починає виконання з `f_lasti`, і знову зупиняється на наступному `yield`.

## Debugging Tip: `inspect.getgeneratorstate()`

```python
import inspect
# GEN_CREATED    — щойно створено, ще не запускалось
# GEN_RUNNING    — зараз виконується (ти всередині next/send)
# GEN_SUSPENDED  — зупинено на yield, чекає next/send
# GEN_CLOSED     — виконання завершено або викинуто виняток
```

---

## Генератор vs Корутина: Виробник vs Споживач

Обидва використовують `yield`. Різниця — у **напрямку руху даних**:

```
ГЕНЕРАТОР (producer):             КОРУТИНА (consumer):

   [Generator] ──yield──► [for]      [.send()] ──► [Coroutine]
   Виробляє дані назовні             Споживає дані ззовні

   def gen():                         def coro():
       yield 1    ← дає              while True:
       yield 2                            x = yield    ← отримує
       yield 3
```

### Ключова різниця у синтаксисі

```python
# ГЕНЕРАТОР: yield стоїть SAM або справа від =
def producer():
    yield 42          # дає 42 назовні
    result = yield 10 # дає 10 назовні, потім отримає .send() всередину

# КОРУТИНА: yield у виразі, зліва від =
def consumer():
    while True:
        data = yield  # отримує значення через .send()
```

In [ ]:
def classic_receiver():
    """Класична корутина: споживає значення через .send()."""
    print("[CORO] Готова до прийому значень")
    total = 0
    count = 0
    while True:
        # yield ЗУПИНЯЄ тут
        # .send(value) → відновлює + value присвоюється 'data'
        data = yield
        if data is None:
            break
        total += data
        count += 1
        print(f"[CORO] Отримано: {data}  |  sum={total}, count={count}")
    print(f"[CORO] Завершено. Avg = {total/count:.2f}" if count else "[CORO] Завершено")

# ── Крок 1: Створення ────────────────────────────────────────────────────────
import inspect
coro = classic_receiver()
print(f"Стан після створення: {inspect.getgeneratorstate(coro)}")
print(f"Код ще НЕ виконувався!")
print()

# ── Крок 2: Priming (ОБОВ'ЯЗКОВО перед .send()) ──────────────────────────────
# next() виконує код ДО першого yield і ЗУПИНЯЄ там
next(coro)
print(f"Стан після next():    {inspect.getgeneratorstate(coro)}")
print(f"Корутина SUSPENDED — чекає на .send()")
print()

# ── Крок 3: Надсилання даних ─────────────────────────────────────────────────
coro.send(10)
coro.send(20)
coro.send(15)
coro.send(5)

# ── Крок 4: Завершення ───────────────────────────────────────────────────────
# .send(None) змушує корутину виконати break → StopIteration — це нормально
try:
    coro.send(None)
except StopIteration as ex:
    print(ex.__class__.__name__)

print(f"Стан після close:     {inspect.getgeneratorstate(coro)}")

## Шкала виконання: покроковий розбір

```
Зовнішній код              |  classic_receiver()
───────────────────────────|──────────────────────────────────────
coro = classic_receiver()  |  ← об'єкт створено, код НЕ виконувався
                           |
next(coro)                 |  → print("Готова до прийому")
                           |    total = 0, count = 0
                           |    data = yield   ← СТОП. SUSPENDED.
                           |
coro.send(10)              |  → data = 10  (yield = 10)
                           |    total = 10, count = 1
                           |    print("Отримано: 10 ...")
                           |    data = yield   ← СТОП. SUSPENDED.
                           |
coro.send(20)              |  → data = 20
                           |    ...
                           |    data = yield   ← СТОП. SUSPENDED.
                           |
coro.send(None)            |  → data = None
                           |    break
                           |    print("Завершено. Avg = ...")
                           |  ← StopIteration raised → CLOSED
```

## Пастка: Priming без next() → TypeError

```python
coro = classic_receiver()
coro.send(42)  # TypeError: can't send non-None value to just-started generator
               # ↑ Корутина ще не дійшла до yield → нікуди прийняти значення!

# ПРАВИЛЬНО:
coro = classic_receiver()
next(coro)    # ← просуваємо до першого yield
coro.send(42) # ← тепер OK
```

## Автоматичний Priming через декоратор

```python
def coroutine(func):
    """Декоратор: автоматично робить next() при створенні."""
    def wrapper(*args, **kwargs):
        gen = func(*args, **kwargs)
        next(gen)  # прайминг
        return gen
    return wrapper

@coroutine
def auto_primed_receiver():
    while True:
        data = yield
        print(f"Отримано: {data}")

r = auto_primed_receiver()  # next() вже зроблено!
r.send(42)  # OK одразу
```

In [ ]:
def coroutine(func):
    """Декоратор автоматичного прайміингу класичних корутин."""
    def wrapper(*args, **kwargs):
        gen = func(*args, **kwargs)
        next(gen)
        return gen
    return wrapper

@coroutine
def running_average():
    """Корутина: обчислює ковзне середнє."""
    total, count = 0, 0
    while True:
        value = yield (total / count if count else 0)
        if value is None:
            return
        total += value
        count += 1

# Тепер next() не потрібен — декоратор вже зробив
avg = running_average()
print(f"Тип: {type(avg)}")
print()

# Надсилаємо числа і отримуємо середнє після кожного
for num in [10, 20, 30, 40, 50]:
    current_avg = avg.send(num)
    print(f"  Після send({num:2d}):  avg = {current_avg:.1f}")

avg.close()
print()
print("Зверни увагу: yield повертає обчислене значення у send()")
print("Тобто .send() одночасно: відправляє дані ТА отримує відповідь")

## Двонаправленість yield: send AND receive одночасно

```python
value = yield result
```

Цей один рядок робить дві речі:
1. **Передає `result` назовні** — як значення виразу `next(gen)` або `gen.send(...)`
2. **Отримує значення ззовні** — що присвоюється `value` при наступному `.send()`

```
Зовнішній код              |  Корутина
───────────────────────────|───────────────────────────────────────
response = coro.send(10)   |  value = yield result
                           |         │            │
                           |    ←────┘            └────►
                           |  value отримує 10   result іде в response
```

### Порівняння: next() vs send()

```python
gen = my_coroutine()
next(gen)        # ≡ gen.send(None)  → value = None
gen.send(42)     # → value = 42

gen.throw(ValueError, "error")  # кидає виняток ВСЕРЕДИНІ yield
gen.close()      # кидає GeneratorExit ВСЕРЕДИНІ yield
```

In [ ]:
@coroutine
def echo_and_count():
    """Двонаправлена корутина: отримує рядок, повертає його + порядковий номер."""
    count = 0
    while True:
        # yield ОДНОЧАСНО:
        # - повертає (count, останнє_слово) назовні
        # - отримує нове_слово ззовні через .send()
        word = yield count
        if word is None:
            return
        count += 1
        print(f"  [CORO] Отримано слово #{count}: '{word}'")

ec = echo_and_count()

print("Двонаправлений yield: send() → отримуємо відповідь:")
for w in ["hello", "asyncio", "yield", "python"]:
    num = ec.send(w)
    print(f"  [MAIN] send('{w}') → отримав номер: {num}")

ec.close()
print()
print("Кожен .send() одночасно ПЕРЕДАЄ і ОТРИМУЄ дані")
print("Саме це — основа Event Loop: він send()-ить результати назад у корутини")

---

## `yield from` — Делегування та Прозоре Тунелювання

`yield from iterable` — замінник вкладеного циклу, але із суперсилою:
він **прозоро тунелює** `.send()`, `.throw()` та `.close()` між
зовнішнім кодом і вкладеним підгенератором.

### Без yield from (ручне делегування)

```python
def delegator_manual():
    for item in subgenerator():
        yield item          # лише дані, .send() не передається!
```

### З yield from (прозоре делегування)

```python
def delegator():
    result = yield from subgenerator()  # ← ВСЕ тунелюється
    # result = return value subgenerator (StopIteration.value)
```

```
Зовнішній          delegator()       subgenerator()
─────────          ───────────       ──────────────
.send(x)  ──────►  yield from  ──────►  data = yield
                       │
response ◄──────  yield from  ◄──────  yield result
.throw() ──────►  yield from  ──────►  try/except
.close() ──────►  yield from  ──────►  GeneratorExit
```

### Чому це важливо для asyncio?

`yield from` → **`await`**. Коли Python 3.5 ввів `async/await`,
`await expr` — це синтаксичний цукор над `yield from expr`
(з додатковими перевірками типів).

```python
# Стара бібліотека (Python 3.4, asyncio з yield from):
@asyncio.coroutine
def old_style():
    result = yield from asyncio.sleep(1)  # тунелює через Event Loop
    return result

# Сучасна (Python 3.5+):
async def modern():
    result = await asyncio.sleep(1)       # те саме, але чистіше
    return result
```

In [ ]:
# Демонстрація yield from: прозоре делегування

def inner_generator():
    """Підгенератор: виробляє значення і приймає .send()."""
    print("  [INNER] Стартую")
    x = yield "inner_1"
    print(f"  [INNER] Отримав через send: {x}")
    y = yield "inner_2"
    print(f"  [INNER] Отримав через send: {y}")
    return "inner_result"  # ← значення StopIteration

def outer_generator():
    """Делегатор: прозоро тунелює через yield from."""
    print("[OUTER] Перед yield from")
    # result отримає return value з inner_generator
    result = yield from inner_generator()
    print(f"[OUTER] Після yield from, inner повернув: '{result}'")
    yield "outer_end"

gen = outer_generator()

print("=== Демонстрація yield from: прозоре тунелювання ===")
print()

# next() → проходить крізь outer → потрапляє в inner
val1 = next(gen)
print(f"[MAIN] Отримав: '{val1}'")
print()

# .send() → проходить крізь outer yield from → потрапляє в inner
val2 = gen.send("дані_1")
print(f"[MAIN] Отримав: '{val2}'")
print()

# Ще один .send() → inner завершується → outer продовжується
val3 = gen.send("дані_2")
print(f"[MAIN] Отримав: '{val3}' (тепер ми в outer після yield from)")
print()
print("Висновок: .send() прозоро пройшов через outer → inner")
print("Саме так Event Loop 'поглиблюється' у вкладені await")

---

## Міст: від `yield from` до `await`

### Еволюція асинхронного Python

```
Python 2.5  (2006):  .send() + .throw()  — перша корутина через yield
Python 3.4  (2014):  yield from           — делегування + asyncio
Python 3.5  (2015):  async def / await   — синтаксичний цукор
Python 3.10 (2021):  match + async       — доповнення
```

### Що таке await під капотом?

```python
# await X  ≡  yield from X.__await__()
# (де X — будь-який Awaitable: coroutine, Task, Future)

await asyncio.sleep(1)
# ↕
yield from asyncio.sleep(1).__await__()
# ↕ (спрощено)
# Event Loop отримує контроль через yield
# Через 1s: Event Loop робить .send(result) → відновлює корутину
```

### Хто викликає .send()?

```
Ти пишеш:                     Event Loop (всередині) робить:
─────────────────────────      ─────────────────────────────────────
await asyncio.sleep(1)    →    loop.call_later(1, task.send, None)
                               # через 1s:
                               task.send(None)  ← Loop сам викликає!
```

Ось відповідь на "питання для роздумів":
> **Event Loop** відповідає за виклик `.send()` — він і є тим
> "зовнішнім кодом", який відновлює призупинені корутини.

### Практичний підсумок

| Рівень | Синтаксис | Хто відновлює |
|--------|-----------|---------------|
| Класична корутина | `data = yield` | Ти вручну через `.send()` |
| Python 3.4 asyncio | `result = yield from future` | Event Loop через `.send()` |
| Сучасний asyncio | `result = await coro` | Event Loop через `.send()` |
| Різниця | Тільки синтаксис | Механізм однаковий |

In [ ]:
# Демонстрація: власний Awaitable — показує що await = yield from
import asyncio

class SimpleAwaitable:
    """Мінімальний awaitable об'єкт — реалізує __await__."""

    def __init__(self, delay):
        self.delay = delay

    def __await__(self):
        # __await__ повертає ітератор
        # Event Loop отримує контроль через yield
        print(f"  [AWAITABLE] Кажу Event Loop: 'зачекай {self.delay}s'")
        yield  # ← ось де await відпускає Event Loop
        # Після того як Event Loop нас розбудить:
        print(f"  [AWAITABLE] Event Loop розбудив нас!")
        return f"result_after_{self.delay}s"

async def my_coroutine():
    print("[CORO] Стартую")
    # await = yield from SimpleAwaitable(1).__await__()
    result = await SimpleAwaitable(0.1)
    print(f"[CORO] Отримав result: '{result}'")
    return "done"

async def main():
    print("=== await під капотом ===")
    print()
    # asyncio.run() створює Event Loop
    # Event Loop викликає .send(None) на coroutine
    # Coroutine доходить до yield в __await__ → повертає контроль Loop
    # Loop реєструє timer → через delay: loop.send(None) → продовжує
    result = await my_coroutine()
    print(f"[MAIN] Coroutine завершилась: '{result}'")

await main()

## Підсумок Частини I: yield → coroutine → await

```
yield                     ← призупинення + збереження стану
    │
    ▼
data = yield              ← двонаправлений: отримуємо через .send()
    │
    ▼
result = yield from sub() ← прозоре делегування (Python 3.4)
    │
    ▼
result = await coro()     ← сучасний синтаксис (Python 3.5+)
    │                       механізм той самий: yield from __await__()
    ▼
Event Loop                ← той, хто викликає .send() автоматично
```

### Ключові висновки

1. **`yield` зберігає стан** — frame object у heap (~1 KB на корутину)
2. **`.send(value)`** відновлює + передає значення всередину
3. **`yield from`** тунелює `.send()/.throw()/.close()` прозоро
4. **`await`** = `yield from x.__await__()` із syntax sugar
5. **Event Loop** = той, хто систематично викликає `.send()` на всіх Tasks

---

# Частина II — Сучасний Asyncio: Event Loop та Coroutines

---

## Event Loop під капотом: як ОС повідомляє про готові сокети

До `async/await` ми бачили, як Event Loop **викликає `.send()`** щоб відновити корутину.
Але звідки він знає, *коли* це робити? Відповідь: **системні виклики ОС**.

### Еволюція: select → poll → epoll / kqueue

```
1983  select(fd_set)   — до 1024 сокетів, O(n) сканування → повільно
1997  poll(pollfd[])   — необмежено, але всі ще O(n)
2002  epoll (Linux)    — O(1)! Тільки готові дескриптори
2000  kqueue (macOS)   — аналог epoll для BSD/macOS
2006  IOCP (Windows)   — completion-based (asyncio.ProactorEventLoop)
```

### Схема взаємодії ОС ↔ Event Loop

```
┌─────────────────────────────────────────────────────────────────────┐
│  asyncio Event Loop (Python, 1 потік)                               │
│                                                                     │
│  while True:                                                        │
│    1. Виконати всі READY tasks (до першого await)                  │
│    │                                                                │
│    2. ─── SYSCALL ──►  epoll_wait(epfd, events, timeout)           │
│    │                   ОС ПЕРЕХОДИТЬ У РЕЖИМ ОЧІКУВАННЯ            │
│    │                   (Python thread заблокований у ядрі ОС!)     │
│    │                                                                │
│    │    (мережевий пакет прийшов → NIC → DMA → ядро → wake up)    │
│    │                                                                │
│    3. ◄── ПОВЕРНЕННЯ ─  [(sock3, EPOLLIN), (sock7, EPOLLOUT), ...] │
│    │                   Список ТІЛЬКИ готових дескрипторів           │
│    │                                                                │
│    4. Для кожного готового fd:                                      │
│         future = fd_to_future[fd]                                   │
│         future.set_result(data)   ← "результат готовий"            │
│         task = future._callbacks  ← Task що чекав на цей Future    │
│         ready_queue.append(task)  ← ставимо в чергу виконання      │
└─────────────────────────────────────────────────────────────────────┘
```

### Ключова думка

> Event Loop **не чекає** у Python-коді — він передає управління ядру ОС через
> `epoll_wait()`. Ядро прокидає потік лише коли є реальна подія (дані в сокеті,
> таймер спрацював). Це і є **справжня** ефективність asyncio — нуль CPU waste.


## Future та Task: два ключових об'єкти asyncio

### `asyncio.Future` — "обіцянка результату"

```python
# Future — низькорівневий об'єкт: результату ще немає, але буде

future = asyncio.Future()

# Стани Future:
# ┌──────────┐  set_result()   ┌──────────┐
# │ PENDING  │ ──────────────► │   DONE   │
# └──────────┘                 └──────────┘
#      │         set_exception()     │
#      └──────────────────────────── ┘

future.done()          # False поки результату немає
future.set_result(42)  # встановлює результат
future.result()        # 42 — отримати результат
```

**Future пов'язує сокет і корутину:**
- Event Loop реєструє: "коли сокет `fd=7` готовий → `future.set_result(data)`"
- Корутина робить: `data = await future` (чекає на результат)
- Коли дані прийшли → `future.set_result()` → корутина прокидається

---

### `asyncio.Task(Future)` — "корутина під управлінням Loop"

```python
# Task успадковує Future + обгортає корутину

task = asyncio.create_task(my_coroutine())

# Task внутрішньо:
class Task(Future):
    def __init__(self, coro):
        self._coro = coro          # корутина що виконується
        self._step()               # запускає перший .send(None)

    def _step(self, result=None):
        try:
            # Відновлює корутину, передаючи result
            future_or_value = self._coro.send(result)
            # Корутина знову заснула на await → future_or_value = inner Future
            future_or_value.add_done_callback(self._wakeup)
        except StopIteration as e:
            self.set_result(e.value)  # корутина завершилась

    def _wakeup(self, inner_future):
        # Викликається коли inner_future готовий
        self._step(inner_future.result())  # .send(result) → відновлення
```

### Ієрархія об'єктів

```
asyncio.Future          — "є результат / очікуємо результату"
    │
    └── asyncio.Task    — "виконує корутину, поки та не поверне результат"
                           Task._step() ←→ coro.send() ←→ yield ←→ await
```


In [ ]:
import asyncio

# ── Демонстрація Future вручну ─────────────────────────────────────────────
async def waiter(future):
    print(f"[WAITER] Чекаю на Future... (стан: {future._state})")
    result = await future  # ← зупинится тут, поки future.set_result()
    print(f"[WAITER] Отримав результат: {result!r}")
    return result

async def resolver(future):
    await asyncio.sleep(0.3)  # Симуляція: "мережевий пакет прийшов"
    print(f"[RESOLVER] Встановлюю результат у Future...")
    future.set_result("дані з мережі")  # ← розбуджує waiter

async def main():
    # Вручну створюємо Future (зазвичай Event Loop робить це автоматично)
    loop = asyncio.get_event_loop()
    future = loop.create_future()

    print(f"Future створено: done={future.done()}")
    print()

    # Запускаємо обидві корутини concurrent
    await asyncio.gather(waiter(future), resolver(future))

    print()
    print(f"Future після завершення: done={future.done()}, result={future.result()!r}")
    print()
    print("Ось що відбулось під капотом:")
    print("  1. waiter() дійшов до 'await future' → yield → повернув Future у Task._step()")
    print("  2. Task._step() зареєстрував callback: future.add_done_callback(task._wakeup)")
    print("  3. resolver() викликав future.set_result() → спрацював callback")
    print("  4. task._wakeup() → task._step(result) → waiter() відновився з результатом")

await main()


## Покрокове: що відбувається при `result = await db.fetch("SELECT ...")`

```
Твій код                  asyncio Task             asyncio Future       ОС (epoll)
─────────────────         ────────────────         ──────────────       ──────────

result = await            task._step(None)
  db.fetch("SELECT")       │
  │                        │
  └─► coroutine            coro.send(None)
      запускається         │
      │                    │
      (asyncpg внутрішньо) │
      ├─ socket.connect()  │                        future = Future()
      ├─ send(SQL bytes)   │                        epoll.register(fd,
      └─ yield future ─────┴──────────────────────► EPOLLIN)
         (SUSPENDED)                                # "коли дані прийдуть
                                                    # → future.set_result()"
         ... Event Loop тепер вільний! ...
         ... Може обслуговувати інші корутини ...

(PostgreSQL обробив запит, мережевий пакет прийшов до NIC)

                                                    epoll_wait() повертає:
                                                    [(fd=7, EPOLLIN)]
                           task._wakeup()  ◄───────  future.set_result(rows)
                           task._step(rows)
                           │
                           coro.send(rows)
                           │
result = rows  ◄───────────┘
(RESUMED)

# Наступний рядок твого коду виконується:
return [User(**row) for row in result]
```

### Висновок: Event Loop = "планувальник з epoll-сердцем"

```
Корутина:     "не знаю коли дані прийдуть → засинаю на Future"
asyncpg:      "реєструю сокет у epoll, прив'язую до Future"
Event Loop:   "питаю epoll які сокети готові → будю відповідні Tasks"
ОС (epoll):   "чекаю апаратних переривань, говорю Loop про готові дескриптори"
NIC/kernel:   "пакет прийшов → DMA → ring buffer → interrupt → epoll event"
```

> Вся ця машинерія — за лаштунками кожного `await` у твоєму коді.
> Ти пишеш `await conn.fetch()`, Python/asyncpg/ОС роблять решту.


# Learning Objectives

Після цього уроку ти зможеш:

## Conceptual Understanding

- Пояснити що таке **coroutine** і чому виклик `async def` не виконує код
- Описати як **Event Loop** працює як кооперативний планувальник задач
- Пояснити різницю між `await asyncio.sleep()` та `time.sleep()`
- Описати механізм **cooperative multitasking** vs **preemptive multitasking**
- Пояснити де `asyncio` виграє, а де потрібен `multiprocessing`

## Debugging Skills

- Передбачити вивід коду з coroutines та `gather()`
- Розпізнати "swallowed exceptions" у background Tasks
- Знайти blocking код у async контексті через `PYTHONASYNCIODEBUG=1`
- Пояснити чому `RuntimeWarning: coroutine was never awaited` виникає

## Production Skills

- Правильно використовувати `asyncio.create_task()` для concurrent виконання
- Розуміти `async with` та `async for`
- Делегувати CPU-bound задачі через `run_in_executor()`
- Будувати async FastAPI ендпоінти без блокування Event Loop

# Ментальна модель: Шеф-кухар і Event Loop

Щоб зрозуміти `asyncio`, зупинись і уяви **одного дуже ефективного шеф-кухаря** в ресторані.

```
Синхронний (blocking) підхід:
┌──────────────────────────────────────────────────────┐
│  Кухар кладе хліб у тостер                          │
│  👀 ...чекає 2 хвилини дивлячись на тостер...       │
│  Хліб готовий                                        │
│  Кухар починає різати цибулю                        │
└──────────────────────────────────────────────────────┘

Асинхронний (asyncio) підхід:
┌──────────────────────────────────────────────────────┐
│  Кухар кладе хліб у тостер → await (ставить таймер)│
│  Відразу йде різати цибулю (для іншого замовлення)  │
│  Таймер спрацював → Event Loop прокидає кухаря      │
│  Кухар повертається до тосту — точно звідки пішов   │
└──────────────────────────────────────────────────────┘
```

**Один кухар (один потік), десятки замовлень (тисячі з'єднань).**

## Масштаб часу: Чому I/O так повільний

| Операція | Реальний час | "Людський" час |
|----------|-------------|----------------|
| CPU cache hit | 1 нс | 3 секунди |
| RAM access | 100 нс | 5 хвилин |
| SSD read | 100 мкс | 4 дні |
| TCP round trip | 100 мс | 11 місяців |
| External API call | 500 мс | **5.5 років** |

> Синхронний сервер змушує потік **чекати 5.5 років** між двома рядками Python.
> Asyncio за цей час обслуговує тисячі інших з'єднань.

---

# Вправа 1 — Забутий await: Найчастіша помилка Asyncio

## Хибне припущення

> Виклик `async def` функції запускає її виконання, як і звичайну функцію

## Передбач результат

Подивись на код. **Не запускай.** Що виведе програма?

- А) `"Starting..."` (перший `print` виконується)
- Б) `"Starting..."` і через секунду `"Finished"`
- В) `RuntimeWarning` і нічого не виведеться

In [ ]:
import asyncio

async def fetch_data():
    print("Starting...")
    await asyncio.sleep(1)
    print("Finished")

# Просто викликаємо функцію, як звичайну
result = fetch_data()

# Що таке result?
print(f"Тип result: {type(result)}")
print(f"Значення result: {result}")

# Програма завершується — що станеться з об'єктом?
# Python garbage collector знищить його і видасть попередження

## Що відбулось?

```
fetch_data()  →  Не виконує код!
              →  Повертає <coroutine object fetch_data at 0x...>
              →  "Starting..." ніколи не виводиться
              →  GC знищує об'єкт → RuntimeWarning: coroutine 'fetch_data' was never awaited
```

## Lifecycle coroutine об'єкта

```
          async def fetch_data()  ← Визначення функції
                    │
          fetch_data()            ← Виклик функції (НЕ виконує!)
                    │
    ┌─────────────────────────┐
    │  coroutine object       │  ← СТАН: GEN_CREATED (заморожений)
    │  code: рядки 3-5        │
    │  locals: {}             │
    │  state: CREATED         │
    └─────────────────────────┘
                    │
          await fetch_data()  або  asyncio.run(fetch_data())
                    │
    ┌─────────────────────────┐
    │  СТАН: GEN_RUNNING      │  ← Виконується
    │  print("Starting...")   │
    │  await asyncio.sleep(1) │  ← ПАУЗА, СТАН: GEN_SUSPENDED
    │  print("Finished")      │  ← Відновлення, СТАН: GEN_CLOSED
    └─────────────────────────┘
```

## 3 способи запустити coroutine

```python
# 1. asyncio.run() — точка входу програми
asyncio.run(fetch_data())

# 2. await — всередині іншої async функції
async def main():
    await fetch_data()

# 3. asyncio.create_task() — планує виконання, повертає Task
async def main():
    task = asyncio.create_task(fetch_data())
    await task
```

## Debugging Tip

```python
# Ввімкни debug режим — знаходить "забуті" coroutines
asyncio.run(main(), debug=True)

# Або через змінну середовища (у терміналі):
# PYTHONASYNCIODEBUG=1 python my_script.py
```

In [ ]:
import asyncio

async def fetch_data():
    print("Starting...")
    await asyncio.sleep(1)
    print("Finished")

async def main():
    print("=== Правильний запуск coroutine ===")
    # Правильно: await виконує coroutine і чекає результату
    await fetch_data()
    print()
    
    print("=== create_task: планує без blocking ===")
    # create_task планує виконання, але НЕ чекає
    task = asyncio.create_task(fetch_data())
    print(f"Task створено, done={task.done()}")
    # Треба await task або main завершиться раніше
    await task
    print(f"Task завершено, done={task.done()}")

asyncio.run(main())

---

# Вправа 2 — Ілюзія Concurrency: time.sleep vs asyncio.sleep

## Хибне припущення

> `asyncio.gather()` магічним чином робить функції паралельними

## Передбач результат

Що виведе цей код — чи буде `health_ping` виводити "Ping!" під час роботи `background_task`?

- А) Так — gather запускає їх паралельно, Ping! перемежовується з Task
- Б) Ні — спочатку 5 секунд тиші, потім Task finished, потім усі Ping! одразу
- В) Програма аварійно завершиться

In [ ]:
import asyncio
import time

async def background_task():
    print("[TASK] Started")
    # ПОМИЛКА: звичайний time.sleep блокує весь Event Loop!
    time.sleep(3)
    print("[TASK] Finished")

async def health_ping():
    for i in range(4):
        print(f"[PING] Ping #{i+1}!")
        await asyncio.sleep(0.5)

async def main():
    print("Запускаємо gather()...")
    await asyncio.gather(background_task(), health_ping())
    print("Завершено")

asyncio.run(main())

## Пояснення: Чому time.sleep зупиняє весь Event Loop

```
[T=0]   Event Loop запускає background_task()
[T=0]   time.sleep(3) → OS ЗУПИНЯЄ PYTHON THREAD (не лише задачу!)
        ╔══════════════════════════════════════════════╗
        ║  EVENT LOOP ЗАМОРОЖЕНИЙ                      ║
        ║  health_ping — чекає, нічого не робить       ║
        ║  Всі інші з'єднання — заморожені             ║
        ╚══════════════════════════════════════════════╝
[T=3]   Thread прокидається, background_task завершується
[T=3]   Event Loop нарешті може запустити health_ping
[T=3.5] Ping #1, Ping #2, Ping #3, Ping #4 — один за одним
```

## Правильний варіант

In [ ]:
import asyncio

async def background_task_correct():
    print("[TASK] Started")
    # ПРАВИЛЬНО: await asyncio.sleep() звільняє Event Loop
    await asyncio.sleep(3)
    print("[TASK] Finished")

async def health_ping():
    for i in range(6):
        print(f"  [PING] Ping #{i+1}!")
        await asyncio.sleep(0.5)

async def main():
    print("З await asyncio.sleep() — справжнє cooperative multitasking:")
    await asyncio.gather(background_task_correct(), health_ping())

asyncio.run(main())

print()
print("Бачиш? TASK і PING перемежовуються!")
print("Ключове слово await = 'відпусти Event Loop, повернись пізніше'")

---

# Що відбувається під капотом: Event Loop та Cooperative Multitasking

## Два типи планування задач

| | Preemptive (Threading/OS) | Cooperative (Asyncio) |
|--|--------------------------|----------------------|
| Хто перериває | ОС примусово | Coroutine добровільно |
| Коли перемикання | Будь-який момент | Лише при `await` |
| Race conditions | Так (треба Lock) | Ні (один потік) |
| Пам'ять | Спільна (небезпечно) | Ліниво-ізольована |
| Overhead | Context switch (дорого) | Мінімальний |

## Архітектура Event Loop

```
┌─────────────────────────────────────────────────────────────┐
│                     EVENT LOOP                              │
│                                                             │
│  ┌───────────────┐    ┌──────────────────────────────────┐ │
│  │  Task Queue   │    │  I/O Selector (epoll/kqueue)     │ │
│  │               │    │                                  │ │
│  │  [Task A] ──► │    │  Socket 1: HTTP ── waiting       │ │
│  │  [Task B]     │    │  Socket 2: DB  ── waiting        │ │
│  │  [Task C]     │    │  Socket 3: API ── DATA READY! ◄─ │ │
│  └───────────────┘    └──────────────────────────────────┘ │
│          │                          │                        │
│          ▼                          ▼                        │
│    Виконати Task           Перемістити Task в Queue         │
│    до першого await                                          │
└─────────────────────────────────────────────────────────────┘
```

## Алгоритм Event Loop (спрощено)

```python
# Псевдокод роботи Event Loop
while True:
    # 1. Виконати всі задачі з черги до першого await
    for task in ready_queue:
        task.step()  # виконати до await

    # 2. Запитати ОС: "Які сокети готові?"
    ready_sockets = selector.select(timeout=next_timer)

    # 3. Перемістити чекаючі задачі в чергу
    for sock in ready_sockets:
        waiting_task = socket_to_task[sock]
        ready_queue.append(waiting_task)
    
    # 4. Перевірити таймери (asyncio.sleep)
    for timer in expired_timers:
        ready_queue.append(timer.task)
```

---

# asyncio.gather та asyncio.create_task

## Різниця між gather і create_task

```
asyncio.gather(*coros):
  1. Приймає coroutines або Tasks
  2. Автоматично загортає в Tasks
  3. Чекає поки ВСІ завершаться
  4. Повертає список результатів у тому ж порядку
  5. Якщо одна впала → всі отримують виняток (якщо return_exceptions=False)

asyncio.create_task(coro):
  1. Планує виконання ЗАРАЗ (не чекає await)
  2. Повертає Task об'єкт негайно
  3. Корисно для fire-and-forget з явним контролем
  4. УВАГА: якщо main() завершується раніше Task — Task зникає!
```

In [ ]:
import asyncio
import time

async def simulate_db_query(query_id, delay):
    print(f"  DB Query {query_id}: починаємо...")
    await asyncio.sleep(delay)
    result = f"data_from_query_{query_id}"
    print(f"  DB Query {query_id}: завершено за {delay}s")
    return result

async def main():
    print("=== Послідовне виконання (без gather) ===")
    start = time.perf_counter()
    r1 = await simulate_db_query(1, 1.0)
    r2 = await simulate_db_query(2, 0.8)
    r3 = await simulate_db_query(3, 0.6)
    seq_time = time.perf_counter() - start
    print(f"Послідовно: {seq_time:.2f}s, результати: {[r1,r2,r3]}")
    
    print()
    print("=== asyncio.gather (concurrent) ===")
    start = time.perf_counter()
    # Всі три стартують ОДНОЧАСНО, чекаємо найповільнішу
    results = await asyncio.gather(
        simulate_db_query(1, 1.0),
        simulate_db_query(2, 0.8),
        simulate_db_query(3, 0.6)
    )
    gather_time = time.perf_counter() - start
    print(f"Gather: {gather_time:.2f}s (очікували ~1.0s), результати: {results}")
    print(f"Прискорення: {seq_time/gather_time:.1f}x")

asyncio.run(main())

---

# Пастка: Silent Task Failures (Зниклі Винятки)

## Найнебезпечніший edge case asyncio

Якщо ти створюєш Task через `create_task()` і **ніколи не `await`-уєш** — виняток зникає мовчки.

```python
async def broken_task():
    raise ValueError("Щось пішло не так!")

async def main():
    # Створюємо Task, але не чекаємо
    asyncio.create_task(broken_task())
    await asyncio.sleep(0)  # Дозволяємо Task виконатись
    print("Main завершився!")  # Це виведеться!
    # ValueError назавжди зник у Task об'єкті
```

## Три правила безпечних Tasks

In [ ]:
import asyncio

async def risky_operation(op_id):
    await asyncio.sleep(0.1)
    if op_id == 2:
        raise RuntimeError(f"Operation {op_id} failed!")
    return f"result_{op_id}"

# ─── НЕБЕЗПЕЧНО: Exception мовчки зникає ───────────────────────────────────
async def unsafe_main():
    print("=== НЕБЕЗПЕЧНО: fire-and-forget без обробки ===")
    # Якщо task кидає виняток і нікому не належить — Python попереджає
    task = asyncio.create_task(risky_operation(2))
    await asyncio.sleep(0.5)
    # task.done() == True, але виняток заморожено всередині
    print(f"Task done: {task.done()}")
    # Якщо не викликати task.result() — виняток зникне при GC

# ─── ПРАВИЛЬНО: завжди retrieve result ────────────────────────────────────
async def safe_main():
    print("=== ПРАВИЛЬНО: обробка виняткі у Tasks ===")
    
    tasks = [asyncio.create_task(risky_operation(i)) for i in range(4)]
    
    # Метод 1: gather з return_exceptions=True
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            print(f"  Task {i}: ПОМИЛКА — {result}")
        else:
            print(f"  Task {i}: {result}")

asyncio.run(unsafe_main())
print()
asyncio.run(safe_main())

---

# Debug Mode: Пошук Прихованих Проблем

## Коли вмикати debug=True

| Ситуація | Що виявляє Debug Mode |
|----------|-----------------------|
| Розробка | Coroutines, що ніколи не awaited |
| Staging | Blocking функції > 100ms |
| Production bugs | Exceptions у background tasks |

```python
# Спосіб 1: через asyncio.run
asyncio.run(main(), debug=True)

# Спосіб 2: через змінну середовища (у терміналі)
# PYTHONASYNCIODEBUG=1 python app.py

# Спосіб 3: програматично
loop = asyncio.get_event_loop()
loop.set_debug(True)
```

## Що виведе debug mode

In [ ]:
import asyncio
import logging

# Увімкнути логування для asyncio
logging.basicConfig(level=logging.DEBUG)

async def slow_coroutine():
    # Симуляція повільної операції без await (блокуюча!)
    import time
    time.sleep(0.2)  # Debug mode виявить це!
    return "done"

async def main():
    await slow_coroutine()

# Debug mode виведе попередження:
# "Executing <coroutine slow_coroutine> took 0.200 seconds"
# "slow_coroutine was never yielded to the event loop"
try:
    asyncio.run(main(), debug=True)
except Exception as e:
    print(f"Error: {e}")
finally:
    # Прибираємо debug logging
    logging.basicConfig(level=logging.WARNING)
    print()
    print("Debug mode виявляє blocking функції > 100ms!")
    print("PYTHONASYNCIODEBUG=1 — вмикай завжди на розробці")

---

# Production: asyncio в FastAPI та Backend-системах

## Три сценарії FastAPI endpoint

```python
from fastapi import FastAPI
import asyncio, time

app = FastAPI()

# Сценарій А: async def + правильний await
@app.get("/async-correct")
async def get_data_async():
    await asyncio.sleep(2)  # Не блокує інших користувачів
    return {"message": "OK"}

# Сценарій Б: async def + БЛОКУЮЧИЙ виклик (найнебезпечніше!)
@app.get("/async-broken")
async def get_data_broken():
    time.sleep(2)           # Блокує ВЕСЬ сервер!
    return {"message": "OK"}

# Сценарій В: def (не async)
@app.get("/sync-safe")
def get_data_sync():
    time.sleep(2)           # FastAPI розпізнає def → threadpool
    return {"message": "OK"}  # Не блокує event loop
```

## Шкала виконання для 1000 клієнтів

```
Сценарій А (async + await):
  [T=0ms]   1000 запитів → 1000 coroutines стартують
  [T=0.5ms] Всі 1000 корутин чекають на await
  [T=2000ms] Всі 1000 прокидаються, відповідають
  Throughput: ~500 req/s

Сценарій Б (async + blocking):
  [T=0ms]   Запит #1 → time.sleep(2) → Event Loop ЗАМЕРЗ
  [T=2000ms] Запит #1 завершено → Запит #2 стартує
  Throughput: 0.5 req/s (1000x гірше!)

Сценарій В (sync def → threadpool):
  FastAPI автоматично детектує def
  Запускає в окремому потоці (threadpool)
  Event Loop вільний
  Throughput: ~40 req/s (за кількістю потоків)
```

---

# Вправа 3 — Пастка бази даних

## Хибне припущення

> Якщо функція `async def`, то код всередині — автоматично асинхронний

## Передбач результат

Що станеться якщо 10 клієнтів одночасно зроблять запит до цього ендпоінту?

- А) FastAPI автоматично перетворить синхронний запит в async → 10 запитів за ~1 секунду
- Б) Запити виконуватимуться concurrent, бо функція `async def`
- В) Сервер заморозиться, запити виконуватимуться послідовно → ~10 секунд

In [ ]:
import asyncio
import time

# Симуляція synchronous DB driver (psycopg2, sqlite3, SQLAlchemy sync)
def synchronous_db_query(user_id):
    time.sleep(1)  # Блокуючий мережевий виклик до PostgreSQL
    return {"id": user_id, "name": f"User {user_id}"}

# НЕБЕЗПЕЧНИЙ ЕНДПОІНТ (симуляція):
async def get_users_broken(user_id):
    # async def НЕ робить це асинхронним!
    users = synchronous_db_query(user_id)  # Блокує весь Event Loop!
    return users

# ПРАВИЛЬНИЙ ВАРІАНТ 1: asyncpg (async PostgreSQL driver)
async def get_users_correct_v1(user_id):
    # Реальний код з asyncpg:
    # async with pool.acquire() as conn:
    #     result = await conn.fetch("SELECT * FROM users WHERE id=$1", user_id)
    await asyncio.sleep(1)  # Симуляція asyncpg
    return {"id": user_id, "name": f"User {user_id}"}

# ПРАВИЛЬНИЙ ВАРІАНТ 2: run_in_executor для legacy синхронного коду
async def get_users_correct_v2(user_id):
    loop = asyncio.get_event_loop()
    # Відправляємо blocking виклик в окремий потік
    users = await loop.run_in_executor(None, synchronous_db_query, user_id)
    return users

# Тест: 5 "одночасних" запитів
async def simulate_concurrent_requests(handler, n=5):
    start = time.perf_counter()
    results = await asyncio.gather(*[handler(i) for i in range(n)])
    elapsed = time.perf_counter() - start
    return elapsed, results

async def main():
    print("=== НЕБЕЗПЕЧНО: sync DB у async def ===")
    t, _ = await simulate_concurrent_requests(get_users_broken)
    print(f"  Час: {t:.2f}s (послідовно! {t/5:.1f}s/запит)")
    
    print()
    print("=== ПРАВИЛЬНО: async DB driver ===")
    t, _ = await simulate_concurrent_requests(get_users_correct_v1)
    print(f"  Час: {t:.2f}s (concurrent! ~1s для всіх 5)")
    
    print()
    print("=== КОМПРОМІС: run_in_executor для legacy ===")
    t, _ = await simulate_concurrent_requests(get_users_correct_v2)
    print(f"  Час: {t:.2f}s (concurrent через threadpool)")

asyncio.run(main())

---

# Вправа 4 — CPU-Bound Заморозка: Де Asyncio Безсилий

## Хибне припущення

> `async def` для CPU-задачі = задача виконається у фоні

## Передбач результат

Чи буде `handle_new_user()` виводити повідомлення поки `compute_heavy_math()` рахує?

- А) Так — gather запускає їх на різних CPU cores
- Б) Ні — CPU-цикл не має `await`, Event Loop не отримує контроль
- В) Частково — перше повідомлення виведеться, решта — ні

In [ ]:
import asyncio
import concurrent.futures
import time

async def compute_heavy_math():
    print("[MATH] Started (CPU-bound task)")
    # CPU-bound цикл БЕЗ await — Event Loop ніколи не отримає контроль!
    total = sum(i * i for i in range(3_000_000))
    print(f"[MATH] Finished: {total}")
    return total

async def handle_new_user():
    print("[USER] Welcome, new user!")

async def main_broken():
    print("=== НЕБЕЗПЕЧНО: CPU-bound у async def ===")
    start = time.perf_counter()
    await asyncio.gather(
        compute_heavy_math(),
        handle_new_user()
    )
    print(f"Час: {time.perf_counter()-start:.2f}s")
    print("'Welcome, new user!' виводиться ПІСЛЯ math!")

# РІШЕННЯ: ProcessPoolExecutor через run_in_executor
def heavy_math_sync():
    return sum(i * i for i in range(3_000_000))

async def compute_offloaded():
    print("[MATH] Started (offloaded до окремого процесу)")
    loop = asyncio.get_event_loop()
    with concurrent.futures.ProcessPoolExecutor() as executor:
        # Event Loop вільний поки процес рахує!
        total = await loop.run_in_executor(executor, heavy_math_sync)
    print(f"[MATH] Finished: {total}")
    return total

async def main_correct():
    print("=== ПРАВИЛЬНО: CPU-bound в ProcessPoolExecutor ===")
    start = time.perf_counter()
    await asyncio.gather(
        compute_offloaded(),
        handle_new_user()
    )
    print(f"Час: {time.perf_counter()-start:.2f}s")
    print("'Welcome!' виводиться ВІДРАЗУ!")

asyncio.run(main_broken())
print()
asyncio.run(main_correct())

---

# async with та async for: Асинхронні Протоколи

## Коли потрібен async with

`async with` — це `__aenter__` та `__aexit__`, які можуть `await`.

```python
# Синхронний with — не може await
with open("file.txt") as f:
    data = f.read()

# Асинхронний — може await при відкритті/закритті
async with aiofiles.open("file.txt") as f:
    data = await f.read()

# DB connection pool (asyncpg):
async with pool.acquire() as conn:
    result = await conn.fetch("SELECT * FROM users")
# При виході: conn автоматично повертається в pool через __aexit__
```

In [ ]:
import asyncio

# Власний async context manager — симуляція DB connection pool
class FakeConnectionPool:
    def __init__(self, max_size=3):
        self.max_size = max_size
        self._semaphore = asyncio.Semaphore(max_size)
        self._counter = 0
    
    async def __aenter__(self):
        await self._semaphore.acquire()  # Чекаємо вільне з'єднання
        self._counter += 1
        conn_id = self._counter
        print(f"  [POOL] З'єднання #{conn_id} відкрито (зайнято: {self.max_size - self._semaphore._value}/{self.max_size})")
        return conn_id
    
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        self._semaphore.release()
        print(f"  [POOL] З'єднання повернено в pool")
        return False

async def db_query(pool, user_id):
    async with pool as conn:
        await asyncio.sleep(0.3)  # Симуляція SQL запиту
        return {"conn": conn, "user_id": user_id}

async def main():
    pool = FakeConnectionPool(max_size=2)  # Максимум 2 одночасних з'єднання
    
    print(f"Pool розмір: 2, одночасних запитів: 5")
    print("Перші 2 отримають з'єднання одразу, решта — чекатимуть:")
    print()
    
    results = await asyncio.gather(*[db_query(pool, i) for i in range(5)])
    print()
    print(f"Всі {len(results)} запитів завершені!")

asyncio.run(main())

---

# Велике порівняння: Threading vs Multiprocessing vs Asyncio

## Архітектурна таблиця

| | Threading | Multiprocessing | Asyncio |
|--|-----------|----------------|---------|
| Потоки/Процеси | N потоків, 1 процес | N процесів | 1 потік, 1 процес |
| GIL | Один на всіх | Власний у кожного | Один (не проблема) |
| Планування | Preemptive (ОС) | Preemptive (ОС) | Cooperative (await) |
| Пам'ять | Спільна | Ізольована | Ізольована |
| Race conditions | Так (треба Lock) | Ні | Ні |
| CPU-bound | ❌ GIL | ✅ Реальний паралелізм | ❌ Блокує loop |
| I/O-bound | ✅ ОК | ✅ ОК | ✅ Найкраще |
| Пам'ять/задачу | ~8 MB/потік | ~50 MB/процес | ~2 KB/корутина |
| Max concurrent | ~100-500 | ~CPU cores | ~100 000+ |
| Ідеально для | Legacy sync I/O | CPU: ML, image | Web, WebSocket, API |

## Дерево рішень

In [ ]:
import asyncio
import threading
import concurrent.futures
import time

def io_task_sync(n):
    time.sleep(0.3)  # Симуляція I/O
    return n * 2

async def io_task_async(n):
    await asyncio.sleep(0.3)  # Асинхронний I/O
    return n * 2

N = 20

# Threading для I/O-bound
start = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=N) as ex:
    results_t = list(ex.map(io_task_sync, range(N)))
threading_time = time.perf_counter() - start

# Asyncio для I/O-bound
async def run_async():
    return await asyncio.gather(*[io_task_async(n) for n in range(N)])

start = time.perf_counter()
results_a = asyncio.run(run_async())
asyncio_time = time.perf_counter() - start

print(f"I/O-bound задачі ({N} x 0.3s):")
print(f"  Sequential (очікувано): {N * 0.3:.1f}s")
print(f"  ThreadPool:   {threading_time:.2f}s  (concurrent через потоки)")
print(f"  Asyncio:      {asyncio_time:.2f}s  (concurrent через Event Loop)")
print()
print("Обидва ~0.3s — concurrent I/O!")
print("Asyncio: менше пам'яті, більше масштабу (100K+ з'єднань)")
print("Threading: краще для legacy sync бібліотек")

---

# Типові Помилки та Debugging

## Помилка 1 — Синхронна HTTP бібліотека у async коді

```python
# НЕБЕЗПЕЧНО: requests — синхронний, блокує Event Loop
async def fetch(url):
    response = requests.get(url)  # Блокує потік!
    return response.json()

# ПРАВИЛЬНО: aiohttp або httpx
async def fetch_correct(session, url):
    async with session.get(url) as response:
        return await response.json()
```

## Помилка 2 — Синхронний DB драйвер у async ендпоінті

```python
# НЕБЕЗПЕЧНО: psycopg2 sync
async def get_user(id):
    conn = psycopg2.connect(...)  # Блокуючий TCP handshake!
    cursor.execute("SELECT ...")  # Блокуючий мережевий запит!

# ПРАВИЛЬНО: asyncpg або SQLAlchemy async
async def get_user_correct(id, pool):
    async with pool.acquire() as conn:
        return await conn.fetchrow("SELECT * FROM users WHERE id=$1", id)
```

## Помилка 3 — CPU-bound задача в Event Loop

```python
# НЕБЕЗПЕЧНО: рахує у головному потоці
async def resize_image(path):
    img = Image.open(path)
    img = img.resize((800, 600))  # CPU-bound! Блокує loop!

# ПРАВИЛЬНО: ProcessPoolExecutor
async def resize_image_correct(path):
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(process_pool, _resize_sync, path)
```

## Помилка 4 — Main завершується раніше Tasks

In [ ]:
import asyncio

# НЕБЕЗПЕЧНО: background task зникає
async def background_work():
    await asyncio.sleep(0.5)
    print("[BG] Важлива робота завершена!")
    return "result"

async def main_broken():
    print("=== НЕБЕЗПЕЧНО: Task зникає ===")
    asyncio.create_task(background_work())  # Планує, але не чекає
    # main() завершується → Event Loop зупиняється → Task знищується!
    print("[MAIN] Завершено (Task ніколи не виконається!)")

async def main_correct():
    print("=== ПРАВИЛЬНО: зберігаємо посилання та await ===")
    task = asyncio.create_task(background_work())
    print("[MAIN] Робота запланована...")
    await asyncio.sleep(0.1)  # Робимо щось ще
    print("[MAIN] Чекаємо завершення background task...")
    await task  # Явно чекаємо
    print("[MAIN] Все завершено!")

asyncio.run(main_broken())
print()
asyncio.run(main_correct())

---

# Практичні Вправи

## Вправа 1: Benchmark — Sequential vs Concurrent (Рівень: Basic)

Реалізуй та заміряй час для 10 "HTTP запитів" (кожен 0.5s):
1. Послідовне виконання
2. asyncio.gather
3. asyncio.create_task

In [ ]:
import asyncio
import time

async def fake_http_request(request_id, delay=0.5):
    await asyncio.sleep(delay)
    return {"id": request_id, "status": 200}

async def main():
    N = 10
    
    # TODO: Варіант 1 — послідовне виконання
    start = time.perf_counter()
    results_seq = []
    for i in range(N):
        r = await fake_http_request(i)
        results_seq.append(r)
    seq_time = time.perf_counter() - start
    print(f"Послідовно:        {seq_time:.2f}s (очікувано {N*0.5:.1f}s)")
    
    # TODO: Варіант 2 — asyncio.gather
    # Впиши своє рішення тут:
    start = time.perf_counter()
    results_gather = await asyncio.gather(*[fake_http_request(i) for i in range(N)])
    gather_time = time.perf_counter() - start
    print(f"asyncio.gather:    {gather_time:.2f}s (очікувано ~0.5s)")
    
    # TODO: Варіант 3 — create_task
    start = time.perf_counter()
    tasks = [asyncio.create_task(fake_http_request(i)) for i in range(N)]
    results_tasks = await asyncio.gather(*tasks)
    tasks_time = time.perf_counter() - start
    print(f"create_task:       {tasks_time:.2f}s")
    
    print(f"\nПрискорення: {seq_time/gather_time:.1f}x!")

asyncio.run(main())

## Вправа 2: Debug Challenge (Рівень: Intermediate)

Знайди та виправ всі баги в коді нижче.

In [ ]:
import asyncio
import time

# BUG HUNT: знайди та виправ всі проблеми

# Bug 1: Функція ніколи не виконується
async def important_task():
    await asyncio.sleep(0.1)
    return "critical data"

async def main_bug1():
    result = important_task()  # BUG: не await!
    print(f"Result: {result}")  # Виведе coroutine object, не дані

# Bug 2: Blocking в async
async def user_service():
    time.sleep(1)  # BUG: блокує Event Loop!
    return "user data"

# Bug 3: Exception зникає мовчки
async def flaky_service():
    await asyncio.sleep(0.1)
    raise ValueError("Service unavailable")

async def main_bug3():
    asyncio.create_task(flaky_service())  # BUG: exception загублено!
    await asyncio.sleep(0.5)
    print("Все OK?")  # Виводиться, але помилка проковтнута

# TODO: виправ кожен баг:
# Bug 1: додай await
# Bug 2: замінити time.sleep на asyncio.sleep
# Bug 3: зберігай task і await його з try/except

async def fixed_bugs():
    # Fix Bug 1:
    result = await important_task()
    print(f"Bug 1 fixed: {result}")
    
    # Fix Bug 3:
    task = asyncio.create_task(flaky_service())
    try:
        await task
    except ValueError as e:
        print(f"Bug 3 fixed: caught {e}")

asyncio.run(fixed_bugs())

## Вправа 3: Async Rate Limiter (Рівень: Advanced)

Реалізуй rate limiter для async HTTP клієнта: не більше N запитів за секунду.

In [ ]:
import asyncio
import time
from collections import deque

class AsyncRateLimiter:
    """Rate limiter: не більше max_calls викликів за period секунд."""
    
    def __init__(self, max_calls: int, period: float = 1.0):
        self.max_calls = max_calls
        self.period = period
        self._calls = deque()
        self._lock = asyncio.Lock()
    
    async def __aenter__(self):
        async with self._lock:
            now = time.monotonic()
            # Видаляємо застарілі записи
            while self._calls and now - self._calls[0] > self.period:
                self._calls.popleft()
            
            # Якщо досягли ліміту — чекаємо
            if len(self._calls) >= self.max_calls:
                wait_time = self.period - (now - self._calls[0])
                if wait_time > 0:
                    await asyncio.sleep(wait_time)
                    # Оновлюємо після очікування
                    now = time.monotonic()
                    while self._calls and now - self._calls[0] > self.period:
                        self._calls.popleft()
            
            self._calls.append(now)
        return self
    
    async def __aexit__(self, *args):
        return False

async def fake_api_call(call_id, limiter):
    async with limiter:
        await asyncio.sleep(0.05)  # Симуляція API
        return f"response_{call_id}"

async def main():
    # Максимум 3 запити за секунду
    rate_limiter = AsyncRateLimiter(max_calls=3, period=1.0)
    
    print("Тест rate limiter: 9 запитів, ліміт 3/сек")
    start = time.perf_counter()
    
    results = await asyncio.gather(
        *[fake_api_call(i, rate_limiter) for i in range(9)]
    )
    
    elapsed = time.perf_counter() - start
    print(f"9 запитів за {elapsed:.2f}s (очікувано ~3.0s для 3 батчів)")
    print(f"Отримано {len(results)} відповідей")

asyncio.run(main())

---

# Summary

## Ключові концепції уроку

| Концепція | Коротко |
|-----------|---------|
| **coroutine object** | `async def func()` не виконує — повертає об'єкт |
| **await** | Призупиняє coroutine, передає контроль Event Loop |
| **Event Loop** | Кооперативний планувальник — один потік, тисячі задач |
| **cooperative multitasking** | Задачі добровільно відпускають контроль через `await` |
| **asyncio.gather** | Запускає N coroutines concurrent, чекає всіх |
| **asyncio.create_task** | Планує виконання, повертає Task негайно |
| **async with/for** | Асинхронні контекстні менеджери та ітератори |
| **swallowed exception** | Task без await → exception зникає мовчки |
| **run_in_executor** | Міст між sync (CPU/legacy) та async світом |

## 5 золотих правил asyncio

1. **Ніколи** `time.sleep()` всередині `async def` — тільки `await asyncio.sleep()`
2. **Ніколи** синхронні DB/HTTP бібліотеки в async ендпоінтах без `run_in_executor`
3. **Завжди** зберігай посилання на Task — інакше exception зникне
4. **Завжди** `asyncio.run(main(), debug=True)` під час розробки
5. CPU-bound задачі **обов'язково** делегуй в `ProcessPoolExecutor`

## Дерево рішень: Яку concurrency обрати?

```
Задача потребує concurrency?
│
├─ I/O-bound (мережа, файли, БД)?
│   ├─ Є async-native бібліотеки (aiohttp, asyncpg)?  → asyncio
│   └─ Тільки sync бібліотеки (requests, psycopg2)?   → ThreadPoolExecutor
│
└─ CPU-bound (математика, ML, обробка зображень)?
    └─ ProcessPoolExecutor (або ray для distributed)
```

## Що далі?

- `asyncio.Queue` — async черги для producer-consumer патернів
- `aiohttp` — async HTTP клієнт та сервер
- `asyncpg` / `SQLAlchemy async` — async PostgreSQL
- FastAPI + uvicorn — production ASGI сервер
- WebSockets через `websockets` + asyncio